# GSE132080 processing decisions — t_79ff033e

This notebook is the executable reconstruction layer for the source-exhaustive OBS/VAR curation of `prism_collection/GSE132080`. It binds accepted source facts to the append-only successors, rollback identities, immutable X, zero-write replay, Collection readback, and terminal execution evidence.

## Accepted source facts

- NCBI GEO `GSE132080` publishes five supplementary files bound in `source_manifest.json` by URL, byte size, Last-Modified, and SHA-256.
- The series has six GEO samples in three gemgroup pairs: GSM3842207–GSM3842209 are the expression libraries mapped to OBS, while GSM3842210–GSM3842212 are the paired sgRNA-barcode libraries and are retained as technical provenance rather than mapped to OBS rows.
- DOI `10.1038/s41587-019-0387-5`, PMID `31932729`, and PMC `PMC7065968` identify the source publication.
- The raw UMI matrix is 33,694 genes × 23,633 barcodes. The published cell-identities table has 23,608 unique cell barcodes; the exact 25-barcode set difference has no identity row and is excluded, not silently dropped.
- The 33,694 source Ensembl IDs are unique; 34 source symbols are duplicated (68 rows). The immutable X axis retains those source symbols, so VAR must preserve row count/order while exposing the unique Ensembl axis explicitly.
- The 128 experimental sgRNAs map exactly from cell `guide_identity` after removing only its redundant leading target token. Published non-targeting controls and `*` unassigned cells have no sequence row and remain `unknown`, never invented.

In [ ]:
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = next(parent for parent in Path.cwd().parents if (parent / 'pyproject.toml').exists())
evidence = root / 'artifacts/schema_audit/real_dataset_curation_20260722/geo_GSE132080/t_79ff033e'
manifest = json.loads((evidence / 'source_manifest.json').read_text())
inspection = json.loads((evidence / 'inspection_report.json').read_text())
plan = json.loads((evidence / 'plan_receipt.json').read_text())
live = json.loads((evidence / 'live_receipt.json').read_text())
execution = json.loads((evidence / 'execution_report.json').read_text())
assert len(manifest['files']) == 5
assert manifest['publication']['doi'] == '10.1038/s41587-019-0387-5'
assert manifest['publication']['pmid'] == '31932729'
assert manifest['publication']['pmc_url'].endswith('/PMC7065968/')
assert [sample['accession'] for sample in manifest['samples']] == [f'GSM38422{i:02d}' for i in range(7, 13)]
expression = [sample for sample in manifest['samples'] if sample['maps_to_obs_sample']]
guides = [sample for sample in manifest['samples'] if not sample['maps_to_obs_sample']]
assert [(sample['gemgroup'], sample['paired_guide_barcode_accession']) for sample in expression] == [(1, 'GSM3842210'), (2, 'GSM3842211'), (3, 'GSM3842212')]
assert [(sample['gemgroup'], sample['paired_expression_accession']) for sample in guides] == [(1, 'GSM3842207'), (2, 'GSM3842208'), (3, 'GSM3842209')]
assert manifest['denominator_accounting']['excluded_unassigned_barcodes'] == 25
assert inspection['invariants']['writes'] == 0
assert plan['status'] == 'PASS' and plan['mode'] == 'plan'
assert plan['registry_counts']['before'] == plan['registry_counts']['after']
manifest['publication'], [(sample['accession'], sample['role']) for sample in manifest['samples']]

## Current live state and accepted reconstruction evidence

The append-only successors are OBS `lhR6Ny3n8QcVeItH0003` and VAR `GJ1HqkBSHfDD1o4m0002`, linked through unchanged X `NEbod0p6ws0H5wug0000` at 23,608 × 33,694. Rollback identities OBS `lhR6Ny3n8QcVeItH0002` and VAR `GJ1HqkBSHfDD1o4m0001` remain retained.

The successor OBS materializes canonical source-backed metadata, explicit per-field `known`/`unknown`/`not_applicable` state, expression-library GSM mapping by gemgroup, partial guide sequences, and source guide phenotypes. It preserves all raw columns, `original_obs_index`, `obs_uuid`, row order, and OBS→X linkage.

The successor VAR preserves all 33,694 rows, source-symbol index (including 68 duplicate-symbol rows), and exact X-axis order while exposing a unique human Ensembl stable-feature axis. The fresh replay is a zero-write no-op with unchanged Artifact/Collection registry counts; both historical Collections retain their exact target member and were not mutated.

In [ ]:
obs = live['member_after']['obs_before']
x = live['member_after']['x']
var = live['member_after']['var_before']
assert (obs['uid'], x['uid'], var['uid']) == ('lhR6Ny3n8QcVeItH0003', 'NEbod0p6ws0H5wug0000', 'GJ1HqkBSHfDD1o4m0002')
assert (obs['hash'], x['hash'], var['hash']) == ('bbIr6mgkuM6qcUxeVjWmKw', 'gbMxw1JnmmLnKzWKTWdC_V', '7xnAhQFWPI6TRcG-wTWoDQ')
assert (obs['key'], x['key'], var['key']) == ('prism_collection/GSE132080/obs.parquet', 'prism_collection/GSE132080/X.h5ad', 'prism_collection/GSE132080/var.parquet')
assert (obs['n_observations'], x['n_observations'], var['n_observations']) == (23608, 23608, 33694)
assert manifest['current_live_triplet']['obs_uid'] == obs['uid']
assert manifest['current_live_triplet']['x_uid'] == x['uid']
assert manifest['current_live_triplet']['var_uid'] == var['uid']
assert (manifest['current_live_triplet']['obs_hash'], manifest['current_live_triplet']['x_hash'], manifest['current_live_triplet']['var_hash']) == (obs['hash'], x['hash'], var['hash'])
assert manifest['rollback_triplet']['obs_uid'] == execution['rollback_identities']['obs']['uid'] == 'lhR6Ny3n8QcVeItH0002'
assert manifest['rollback_triplet']['var_uid'] == execution['rollback_identities']['var']['uid'] == 'GJ1HqkBSHfDD1o4m0001'
assert (manifest['rollback_triplet']['x_uid'], manifest['rollback_triplet']['x_hash']) == (x['uid'], x['hash'])
assert (manifest['rollback_triplet']['obs_hash'], manifest['rollback_triplet']['var_hash']) == ('koT42ZnpMsA4h7YojGR18Q', 'cTno7idkcjb9xcHS7R6c2w')
assert execution['rollback_identities']['obs'] == {'hash': 'koT42ZnpMsA4h7YojGR18Q', 'key': 'prism_collection/GSE132080/obs.parquet', 'uid': 'lhR6Ny3n8QcVeItH0002'}
assert execution['rollback_identities']['var'] == {'hash': 'cTno7idkcjb9xcHS7R6c2w', 'key': 'prism_collection/GSE132080/var.parquet', 'uid': 'GJ1HqkBSHfDD1o4m0001'}
assert live['status'] == 'PASS' and live['mode'] == 'verify' and live['replay_noop'] is True
assert live['member_after']['source_join']['join_mismatch_count'] == 0
assert live['member_after']['var_verdict']['mismatch_count'] == 0
assert live['member_after']['var_verdict']['axis_count_parity'] is True
assert live['member_after']['var_verdict']['axis_order_parity'] is True
assert live['registry_counts']['before'] == live['registry_counts']['after'] == {'artifacts': 28520, 'collections': 38}
assert live['writes'] == {'artifacts': {'obs': [], 'var': []}, 'collection_writes': 0, 'deletions': 0, 'obs_revisions': 0, 'var_revisions': 0, 'x_revisions': 0}
assert live['collections']['historical_manifest_identity'] == 'jkobject:GCjqQtGwPzkY'
assert [(live['collections'][name]['uid'], live['collections'][name]['hash'], live['collections'][name]['member_count'], live['collections'][name]['target_key_matches']) for name in ('pert-gym/additions/20260621', 'pert-gym/canonical/20260621')] == [('kYkBznC2fuGmbUbg0000', 'GXnxligyjv_jQmZGzENIiA', 996, [{'key': 'prism_collection/GSE132080/obs.parquet', 'uid': 'lhR6Ny3n8QcVeItH0000'}]), ('aEWBxMlcWx2d7Cd80000', 'FuT5cWa5QodFHfYbzsyLRw', 1056, [{'key': 'prism_collection/GSE132080/obs.parquet', 'uid': 'lhR6Ny3n8QcVeItH0000'}])]
assert execution['mutate']['status'] == 'POSTWRITE_VERIFIER_FAILED_CLOSED' and execution['mutate']['run_uid'] == 'akfsv9rXOldGPZLQ'
assert (execution['mutate']['new_obs']['uid'], execution['mutate']['new_obs']['hash'], execution['mutate']['new_obs']['key']) == (obs['uid'], obs['hash'], obs['key'])
assert (execution['mutate']['x_unchanged']['uid'], execution['mutate']['x_unchanged']['hash'], execution['mutate']['x_unchanged']['key']) == (x['uid'], x['hash'], x['key'])
assert (execution['mutate']['new_var']['uid'], execution['mutate']['new_var']['hash'], execution['mutate']['new_var']['key']) == (var['uid'], var['hash'], var['key'])
assert execution['verify']['canonical_sha256'] == live['canonical_sha256']
assert execution['verify']['status'] == 'PASS' and execution['verify']['writes'] == live['writes']
assert execution['terminal_control_plane'] == {'instance': 'pert-gym-worker-eu', 'lease_labels_remaining': [], 'local_lease_remaining': False, 'status': 'TERMINATED', 'zone': 'europe-west1-b'}
{'triplet': (obs['uid'], x['uid'], var['uid']), 'shape': (23608, 33694), 'replay_noop': live['replay_noop'], 'writes': live['writes']}